##### Task 7
Build cleaned silver tables.

What I want in the end:
- silver_customers
- silver_items
- silver_orders
- No duplicate primary keys
- Valid dates in orders
- Valid foreign key references
- Valid item prices
- Valid nested order items

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- silver_customers

CREATE OR REPLACE TABLE silver_customers AS
SELECT customer_id, name, email, country, created_at
FROM (
  SELECT customer_id, name, email, country, created_at,
  ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY created_at
  ) rn
  FROM bronze_customers
  WHERE customer_id IS NOT NULL
    AND email IS NOT NULL
) t
WHERE rn = 1;

SELECT *
FROM silver_customers;

In [0]:
%sql
-- silver_items

CREATE OR REPLACE TABLE silver_items AS
SELECT item_id, product_name, price, category
FROM (
  SELECT item_id, product_name, price, category,
  ROW_NUMBER() OVER (
    PARTITION BY item_id
    ORDER BY price DESC
  ) rn
  FROM bronze_items
  WHERE item_id IS NOT NULL
    AND (price IS NOT NULL OR price <= 0)
) t
WHERE rn = 1;

SELECT *
FROM silver_items;

In [0]:
%sql
-- silver_orders STEP 1

CREATE OR REPLACE TEMP VIEW valid_order_items AS
SELECT o.order_id, o.item_id, o.quantity
FROM (
  SELECT order_id, item.item_id item_id, item.quantity quantity
  FROM bronze_orders
  LATERAL VIEW explode(items) AS item
) o
LEFT JOIN silver_items i
  ON o.item_id = i.item_id
WHERE i.item_id IS NOT NULL
  AND o.quantity IS NOT NULL
  AND o.quantity > 0;

CREATE OR REPLACE TEMP VIEW valid_order_items_aggregated AS
SELECT order_id, collect_set(named_struct('item_id', item_id, 'quantity', quantity)) items
FROM valid_order_items
GROUP BY order_id;

SELECT *
FROM valid_order_items_aggregated;

In [0]:
%sql
-- silver_orders STEP 2

CREATE OR REPLACE TABLE silver_orders AS
SELECT t.order_id, t.order_date, t.customer_id, t.status, v.items
FROM (
  SELECT *,
  ROW_NUMBER() OVER (
    PARTITION BY order_id
    ORDER BY order_date ASC
  ) rn
  FROM bronze_orders
  WHERE order_id IS NOT NULL
) t
LEFT JOIN silver_customers c
  ON t.customer_id = c.customer_id
LEFT JOIN valid_order_items_aggregated v
  ON t.order_id = v.order_id
WHERE rn = 1
  AND try_to_date(order_date, 'yyyy-MM-dd') IS NOT NULL
  AND c.customer_id IS NOT NULL
  AND v.items IS NOT NULL;

SELECT *
FROM silver_orders;